# Suite PRS — Parsing

The C++ standards parse source text into Ccpp trees by delegating to tree-sitter-cpp, then rewriting its concrete
syntax tree into Ccpp kinds; a token-level pre-pass handles the module declarations, `export` and `extern template`
that tree-sitter-cpp 0.23 cannot parse. Each case parses constructs and checks the tree (by shape where it matters,
and by the text it prints), then the errors: syntax errors, unsupported syntax, and constructs a standard lacks,
each with its line and column.

In [ ]:
import { readFileSync } from "node:fs";

import { Ccpp17, Ccpp20, Syntax as S } from "@mbse/programs/Ccpp";
import { CcppStandard } from "@mbse/programs/Ccpp/_Standard";
import { Errors, Syntax as F } from "@mbse/programs/Framework";

import { assert, equal, raises } from "./support.js";

const { ParseError } = Errors;
const [CPP26, C23] = [new CcppStandard(2026), new CcppStandard(2023, "C")];
type Standard = { parse(text: string): S.TranslationUnit; print(node: F.Node): string };

function parse(text: string, standard: Standard = CPP26): S.TranslationUnit {
  const unit = standard.parse(text);
  equal(S.LANGUAGE.validate(unit), []);
  return unit;
}

const first = (text: string, standard: Standard = CPP26): any => parse(text, standard).items[0];

/** Parses `text`, checks it prints `expected` (by default itself) and that the print parses to the same tree. */
function sameText(text: string, expected: string | null = null, standard: Standard = CPP26): S.TranslationUnit {
  const unit = parse(text, standard);
  const printed = standard.print(unit);
  assert(printed === (expected ?? text) + "\n", printed);
  assert(F.same(parse(printed, standard), unit));
  return unit;
}

/** `sameText` for a statement, in a function's body. */
function sameStatement(text: string, expected: string | null = null): any {
  const wrap = (t: string) => "void f() {\n    " + t.replaceAll("\n", "\n    ") + "\n}";
  return (sameText(wrap(text), wrap(expected ?? text)).items[0] as any).body.items[0];
}

/** The value initializing the one variable `text` declares. */
const initializer = (text: string, standard: Standard = CPP26): any => first(text, standard).declarators[0].initializer.value;
const kinds = (nodes: readonly F.Node[]) => nodes.map((n) => n.kind().KIND);

function fails(text: string, message: string, line: number, column: number, standard: Standard = CPP26): void {
  raises(ParseError, () => standard.parse(text), message);
  try {
    standard.parse(text);
  } catch (error) {
    const e = error as InstanceType<typeof ParseError>;
    equal([e.line, e.column], [line, column]);
    equal(e.message, `line ${line}, column ${column}: ${message}`);
  }
}

## PRS-01 · The conformance source parses to a valid tree, which prints and parses back the same

In [ ]:
const source = readFileSync("../../conformance/sources/cpp20.cpp", "utf8");
let unit = Ccpp20.parse(source);
equal(S.LANGUAGE.validate(unit), []);
assert(F.same(Ccpp20.parse(Ccpp20.print(unit)), unit) && Ccpp20.print(Ccpp20.parse(Ccpp20.print(unit))) === Ccpp20.print(unit));
const found = new Set([...F.walk(unit)].map((n) => n.kind().KIND));
assert(found.size > 120, String(found.size));
console.log(found.size, "kinds");

## PRS-02 · Names: qualified, template, operator, conversion, destructor and literal operator names

In [ ]:
let name = first("a::b<int>::template c<T>::d x;").specifiers[0].name;
assert(name instanceof S.QualifiedName && !name.global_scope && (name.name as S.Identifier).spelling === "d");
equal(kinds(name.qualifiers), ["Identifier", "TemplateId", "TemplateId"]);
assert(!(name.qualifiers[1] as any).template_keyword && (name.qualifiers[2] as any).template_keyword);
name = first("A::template B<int> x;").specifiers[0].name;
assert(name.name.template_keyword && name.name.name.spelling === "B");
assert(first("::x y;").specifiers[0].name.global_scope);
assert(first("decltype(x)::type y;").specifiers[0].name.qualifiers[0] instanceof S.DecltypeSpecifier);
sameText("auto x = A::template f<int>();");
sameText("X<Ts...> x;");
assert(first("X<Ts...> x;").specifiers[0].name.arguments[0].pattern instanceof S.TypeId);
const operators = first("struct S { S &operator=(S &&); void *operator new[](size_t); bool operator<=>(S) const; "
  + "void operator()(); auto operator co_await(); bool operator not_eq(S); };").specifiers[0].body.items;
equal(operators.flatMap((d: F.Node) => [...F.walk(d)].filter((n) => n instanceof S.OperatorName).map((n: any) => n.operator)),
  ["=", "new[]", "<=>", "()", "co_await", "!="]);
const literal = first('int operator""_km(unsigned long long);').declarators[0].declarator.declarator.name;
assert(literal instanceof S.LiteralOperatorName && literal.suffix === "_km");
const destructor = first("A::~A() {}").declarator.declarator.name;
assert(destructor instanceof S.QualifiedName && destructor.name instanceof S.DestructorName);
sameText("struct S {\n    operator int&() const;\n    operator int&&();\n    explicit operator const char*() const;\n};");
sameText("A::operator int*() const {}\n::A::B::operator bool() {}");
const qualified = first("A::operator int*() const {}").declarator.declarator.name;
assert(qualified instanceof S.QualifiedName && qualified.name instanceof S.ConversionName);
const conversion = first("operator int*() const;").declarators[0].declarator;
assert(conversion.declarator.name.type.declarator instanceof S.PointerDeclarator && conversion.qualifiers[0].keyword === "const");
console.log("ok");

## PRS-03 · Literals keep their spelling

In [ ]:
const values = first("auto a = {1'000ull, 0x1.8p3, 1.5e-3, 0b101, 0x1F, 'c', L'w', u8'x', \"s\\n\", R\"d(r)d\", "
  + "u8R\"(x)\", 1_km, \"s\"_sv, true, false, nullptr, NULL};").declarators[0].initializer.value.items;
equal(kinds(values), ["IntegerLiteral", "FloatingLiteral", "FloatingLiteral", "IntegerLiteral", "IntegerLiteral",
  "CharacterLiteral", "CharacterLiteral", "CharacterLiteral", "StringLiteral", "RawStringLiteral", "RawStringLiteral",
  "UserDefinedLiteral", "UserDefinedLiteral", "BooleanLiteral", "BooleanLiteral", "NullptrLiteral", "IdExpression"]);
assert(values[0].spelling === "1'000ull" && values[6].prefix === "L" && values[7].prefix === "u8");
assert(values[8].text === "s\\n" && values[8].prefix === null);
equal([values[9].prefix, values[9].delimiter, values[9].text], [null, "d", "r"]);
equal([values[10].prefix, values[10].delimiter, values[11].suffix], ["u8", null, "_km"]);
assert(values[13].value === true && values[14].value === false);
equal(kinds(initializer('auto s = "a" L"b" PRIu64 R"(c)";').parts), ["StringLiteral", "StringLiteral", "IdExpression",
  "RawStringLiteral"]);
sameText('auto s = "a" L"b" PRIu64 R"(c)";');
console.log("ok");

## PRS-04 · Expressions: operators, alternative tokens, casts, calls, members, new and delete

In [ ]:
sameStatement("x = a, b, c;");
const comma = sameStatement("x, y, z;").expression;
assert(comma.operator === "," && comma.left.operator === "," && comma.right.name.spelling === "z"); // to the left
assert(initializer("bool b = a and not c or d bitand e;").right.operator === "&");
sameText("bool b = a and not c or d bitand e xor f bitor compl g;", "bool b = a && !c || d & e ^ f | ~g;");
sameStatement("x and_eq 1, x or_eq 2, x xor_eq 3, x not_eq 4;", "x &= 1, x |= 2, x ^= 3, x != 4;");
sameStatement("x = -a + +b * !c / ~d % *p - &q;");
sameStatement("x = ++a + --b - a++ - b--;");
assert(initializer("int x = ++a;") instanceof S.UnaryExpression);
assert(initializer("int x = a++;") instanceof S.PostfixExpression);
sameStatement("x = a ? b : c;");
sameStatement("x = a ?: c;");
assert(initializer("int x = a ?: c;").consequence === null);
sameStatement("x = (int)y + static_cast<int>(y) + int(y) + int{y} + T{y} + unsigned(y);");
assert(initializer("int x = int(y);") instanceof S.FunctionalCastExpression);
assert(initializer("int x = f<int>(y);") instanceof S.CallExpression);
assert(initializer("int x = static_cast<int, int>(y);") instanceof S.CallExpression); // not a cast
sameStatement("x = dynamic_cast<B*>(p), const_cast<char*>(s), reinterpret_cast<long>(p);");
sameStatement("x = f(1, g(2), h<int>(3)), a[1], m[1, 2], o.m, p->m, o.*f, s.template get<0>(), p->~T();");
sameStatement("x = sizeof y + sizeof(y) + sizeof(int) + alignof(int) + __alignof__(int) + sizeof...(Ts);");
sameStatement("x = noexcept(f()) && typeid(y) == typeid(*p);");
assert(initializer("bool b = noexcept(f());") instanceof S.NoexceptExpression);
assert(initializer("auto &t = typeid(y);") instanceof S.TypeidExpression);
fails("auto &t = typeid(y, z);", "typeid takes one operand", 1, 17);
sameStatement("x = new T, new T(1), new T{1}, new int[n][4](), ::new (p) T, new (std::nothrow) T[2]{1, 2};");
const made = initializer("int *x = new int[n][4]{};");
assert(made.type.declarator.size.spelling === "4" && made.type.declarator.declarator.size.name.spelling === "n");
sameStatement("delete p, delete[] q, ::delete r;");
sameStatement("x = co_await y;");
sameStatement("x = ({\n    int y = 1;\n    y;\n});");
assert(initializer("int x = ({ 1; });") instanceof S.StatementExpression);
sameStatement("x = __extension__ 1;");
sameStatement("x = (... + xs) + (xs * ... * 1) + (xs && ...);");
sameStatement("f(xs...), g(std::forward<Ts>(xs)...);");
sameStatement("x = offsetof(S, m);");
assert(initializer("int x = offsetof(S, m);").arguments[0] instanceof S.TypeId);
sameStatement("x = {1, {2, 3}, .a = 4};");
sameText("std::vector<int> v = {1, 2,};");
sameText("int x = defined;");
console.log("ok");

## PRS-05 · Lambdas, captures, fold expressions and requires-expressions

In [ ]:
const lambda = initializer("auto f = [=, &a, b, this, *this, c = 1, &d = e, ...f = g, &...h = i, j..., &k...]"
  + "<typename T> requires C<T> (T t) mutable constexpr noexcept -> T requires D<T> { return t; };");
equal(kinds(lambda.captures), ["DefaultCapture", "SimpleCapture", "SimpleCapture", "ThisCapture", "ThisCapture",
  "InitCapture", "InitCapture", "InitCapture", "InitCapture", "SimpleCapture", "SimpleCapture"]);
const caps = lambda.captures;
assert(caps[1].by_reference && !caps[2].by_reference && caps[4].copy);
assert(caps[6].by_reference && caps[7].pack && caps[8].by_reference);
assert(caps[9].pack && caps[10].pack && caps[10].by_reference);
equal(lambda.declarator.specifiers.map((s: any) => s.keyword), ["mutable", "constexpr"]);
assert(lambda.declarator.exception instanceof S.NoexceptSpecifier && lambda.declarator.requires !== null);
sameText("auto f = [=, &a, b, this, *this, c = 1, &d = e, ...f = g, &...h = i, j..., &k...]<typename T> requires C<T>"
  + " (T t) mutable constexpr noexcept -> T requires D<T> {\n    return t;\n};");
sameText("auto f = [] {};");
sameText("auto f = []() throw() [[nodiscard]] {};", null, Ccpp17);
sameText("auto f = [](auto... x) {\n    return (x + ...);\n};");
raises(ParseError, () => CPP26.parse("auto f = []() & {};"), "unsupported syntax: ref_qualifier");
const requires = first("template <class T> concept C = requires (T t, const T &u) { t.f(); typename T::x; "
  + "{ t.g() } noexcept -> std::same_as<int>; { u.h() }; requires D<T>; };").declaration.constraint;
equal(kinds(requires.requirements), ["SimpleRequirement", "TypeRequirement", "CompoundRequirement",
  "CompoundRequirement", "NestedRequirement"]);
assert(requires.requirements[2].noexcept && requires.requirements[3].return_type === null);
sameText("template <class T>\nconcept C = requires { T::value; } && (sizeof(T) > 1 || D<T>);");
sameText("template <class T> requires A<T> && (B<T> || C<T>) && D\nvoid f(T);");
console.log("ok");

## PRS-06 · Declarations and specifiers, in source order

In [ ]:
const declaration = first("[[nodiscard]] static inline const unsigned long long int [[a]] *x = nullptr, y{1}, z(2);");
equal(kinds(declaration.attributes), ["StandardAttributeSpecifier"]);
equal(declaration.specifiers.map((s: any) => s.keyword ?? s.kind().KIND),
  ["static", "inline", "const", "unsigned", "long", "long", "int", "StandardAttributeSpecifier"]);
equal(declaration.declarators.map((d: any) => d.initializer.kind().KIND),
  ["EqualInitializer", "InitializerList", "ParenthesizedInitializer"]);
sameText("int const volatile x = 1;");
sameText("size_t n;");
assert(first("size_t n;").specifiers[0] instanceof S.NamedTypeSpecifier);
assert(first("wchar_t c;").specifiers[0] instanceof S.PrimitiveTypeSpecifier);
sameText("typedef int T, *P;");
sameText("__extension__ typedef long long ll;");
sameText("using T = typename A::B;");
sameText("extern thread_local int t;");
sameText("constexpr auto x = 1;\nconstinit int y = 2;\nconsteval int f();");
sameText("decltype(auto) x = y;\nC auto z = w;\nC<int> decltype(auto) v = u;");
sameText("struct S {\n    explicit(true) S();\n};");
sameText("__declspec(dllexport) alignas(16) alignas(int) int x;");
sameText("int f() __attribute__((noinline, format(printf, 1, 2)));");
sameText("[[gnu::cold]] void f();");
assert(first("[[gnu::cold]] void f();").attributes[0].attributes[0].namespace === "gnu");
sameText("[[deprecated]];");
assert(first("[[deprecated]];") instanceof S.AttributeDeclaration);
sameText(";");
assert(first(";") instanceof S.EmptyDeclaration);
sameText("static_assert(true);\n_Static_assert(1, \"m\");");
sameText("namespace a = b;\nnamespace c = ::d::e;\nusing namespace std;\nusing std::cout;\nusing enum E;");
sameText('extern "C" int x;\nextern "C++" {\n    int y;\n}');
sameText("namespace a::inline b::c {}\ninline namespace d {}\nnamespace [[deprecated]] e {}\nnamespace {}");
sameText("template void f<int>(int);\ntemplate class X<int>;\nextern template class X<long>;");
assert(first("extern template class X<long>;").extern && !first("template class X<int>;").extern);
raises(ParseError, () => CPP26.parse('int x asm("y");'), "unsupported syntax: gnu_asm_expression");
sameText('asm volatile inline goto("jmp %l0" : : : : done);');
sameText('__asm__("nop");\nasm("mov %1, %0" : "=r" (a) : [in] "r" (b) : "memory", "cc");');
console.log("ok");

## PRS-07 · Declarators nest inside out

In [ ]:
const declarator = first("int (*f)(int);").declarators[0].declarator;
assert(declarator instanceof S.FunctionDeclarator && declarator.declarator instanceof S.ParenthesizedDeclarator);
assert(declarator.declarator.declarator instanceof S.PointerDeclarator);
sameText("int *a[3], (*b)[3], **c, &d = x, &&e = 1, *const f = 0, *g(int), (*h(int))(char);");
sameText("void (*handlers[4])(int);");
sameText("int S::*p = &S::x;\nint A::B::*q;\nint ::C::*r;");
const member = first("int A::B::*q;").declarators[0].declarator;
assert(member.scope.qualifiers[0].spelling === "A" && member.scope.name.spelling === "B");
sameText("int x [[a]], y [[b]][3], f [[c]]();");
sameText("void f(int[*], int[static 3], int[const 4]);", null, C23);
sameText("auto [a, b] = t;\nauto &[c, d] = u;");
sameText("void f(int, ...);\nvoid g(...);\nvoid h(int(*)(int), int (*p)[3] = nullptr);");
sameText("template <class... Ts>\nvoid f(Ts... xs, Ts &&...ys);");
sameText("template <class... Ts>\nvoid f(Ts...);");
sameText("void f() const & noexcept(true) [[a]] -> int;");
sameText("void f() throw(int, char);", null, new CcppStandard(2014));
sameText("void f() requires C<T>;\nvirtual void g() override final;\nvirtual void h() = 0;");
const init = first("struct S { virtual void (f)() = 0; int *p = 0; virtual int *g() = 0; int (*h)() = 0; };").specifiers[0].body.items;
assert(init[0].declarators[0].pure && init[1].declarators[0].initializer.value.spelling === "0");
assert(init[2].declarators[0].pure && !init[3].declarators[0].pure && init[3].declarators[0].initializer !== null);
assert(first("struct A { virtual ~A() = 0; };").specifiers[0].body.items[0].declarators[0].pure);
raises(ParseError, () => CPP26.parse("int (__cdecl *f)(int);"), "unsupported syntax: ms_call_modifier");
raises(ParseError, () => CPP26.parse("void __cdecl f() {}"), "unsupported syntax: ms_call_modifier");
raises(ParseError, () => CPP26.parse("int * __sptr p;"), "unsupported syntax: ms_pointer_modifier");
raises(ParseError, () => CPP26.parse('void f() asm("g");'), "unsupported syntax: gnu_asm_expression");
raises(ParseError, () => CPP26.parse("void f(int g() requires C);"), "unsupported syntax: requires_clause");
raises(ParseError, () => CPP26.parse('void f() { x = asm("nop"); }'), "unsupported syntax: gnu_asm_expression");
raises(ParseError, () => CPP26.parse("auto f = [a::b] {};"), "unsupported syntax: qualified_identifier");
console.log("ok");

## PRS-08 · Statements, labels and their substatements

In [ ]:
const body = first("void f() { switch (x) { case 1: a(); b(); case 2: case 3: c(); default: ; } }").body.items[0].body.items;
equal(kinds(body), ["CaseStatement", "ExpressionStatement", "CaseStatement", "DefaultStatement"]);
assert(body[2].statement instanceof S.CaseStatement && body[2].statement.statement instanceof S.ExpressionStatement);
assert(first("void f() { switch (x) { case 1: } }", new CcppStandard(2023)).body.items[0].body.items[0].statement === null);
sameText("void f() {\n    if (a)\n        b();\n    else if (c) {\n        d();\n    } else\n        e();\n}");
sameText("void f() {\n    if constexpr (A) {}\n    if (int x = 1; x) {}\n    if (auto y = g())\n        h(y);\n}");
sameText("void f() {\n    while (a--)\n        continue;\n    do\n        b();\n    while (c);\n    for (;;)\n"
  + "        break;\n    for (x = 0, y = 1; x < y; ++x) {}\n    for (int i = 0; i < n; ++i) {}\n"
  + "    for (auto &[k, v] : m) {}\n    for (int v : {1, 2}) {}\n    for (T t = u; auto x : t) {}\n}");
sameText("void f() {\n    switch (int m = g(); m) {\n        case 1: {\n            break;\n        }\n    }\n}");
sameText("void f() {\nstart:\n    goto start;\nend:\n    int x;\n    return;\n}");
sameText("void f() {\n    try {\n        throw E();\n    } catch (const E &e) {\n        throw;\n    } catch (...) {}\n}");
sameText("void f() try {} catch (...) {}");
sameText("S::S() try : a(1) {} catch (...) {}");
sameText("struct S {\n    S() try : a(1), b{2} {} catch (...) {}\n};");
sameText("void f() {\n    [[likely]] if (a) {}\n    [[fallthrough]];\n}");
sameText("task f() {\n    co_yield 1;\n    co_return 2;\n    co_return;\n}");
raises(ParseError, () => CPP26.parse("void f() { __try {} __finally {} }"), "unsupported syntax: seh_try_statement");
console.log("ok");

## PRS-09 · Classes, enumerations and templates

In [ ]:
const klass = first("class [[a]] S final : public virtual B, [[b]] C<int>, Ds... "
  + "{ public: int x : 3, y; S() : x(1), y{2}, Ds(3)... {} friend class F; friend T; friend void g(); "
  + "protected: };").specifiers[0];
equal(kinds(klass.attributes), ["StandardAttributeSpecifier"]);
equal(kinds(first("class alignas(8) T {};").specifiers[0].attributes), ["AlignasSpecifier"]);
equal(kinds(first("class __declspec(x) T {};").specifiers[0].attributes), ["DeclspecSpecifier"]);
assert(klass.final);
equal(klass.bases.map((b: any) => [b.access, b.virtual, b.pack]), [["public", true, false], [null, false, false],
  [null, false, true]]);
assert(klass.bases[1].attributes.length > 0);
equal(kinds(klass.body.items), ["AccessSpecifier", "SimpleDeclaration", "FunctionDefinition", "SimpleDeclaration",
  "SimpleDeclaration", "SimpleDeclaration", "AccessSpecifier"]);
assert(klass.body.items[2].initializers[2].pack && klass.body.items[1].declarators[0].bitfield.spelling === "3");
const friends = klass.body.items.slice(3, 6);
assert(friends.every((d: any) => d.specifiers[0].keyword === "friend"));
assert(friends[0].specifiers[1] instanceof S.ClassSpecifier && friends[1].specifiers[1] instanceof S.NamedTypeSpecifier);
sameText("struct S;\nunion U {\n    int i;\n    float f;\n};\nstruct {\n    int x;\n} s;");
sameText("enum E : int;\nenum class F { A, B = 2 };", "enum E : int;\nenum class F {\n    A,\n    B = 2\n};");
sameText("enum { A };", "enum {\n    A\n};");
assert(first("enum { A };").specifiers[0].name === null);
sameText("enum struct G : char {\n    A,\n    // comment\n    B,\n};");
sameText("template <class T, class = void, int N = 3, typename... Ts, template <class> class U = std::vector, "
  + "template <class> class... Vs>\nstruct X;");
sameText("template <>\nstruct X<int> {};");
sameText("template <class T>\ntemplate <class U>\nvoid X<T>::f(U) {}");
console.log("ok");

## PRS-10 · Directives and comments are kept where items are listed

In [ ]:
sameText("#include <a>\n#include \"b.h\"\n#include MACRO\n#define X\n#define Y 1\n#define Z(a, b) a##b\n"
  + "#define V(...) f(__VA_ARGS__)\n#define W(a, ...) a\n#pragma once\n#undef X\n#line 3 \"f\"\n#error stop");
assert(first("#include MACRO\n").macro instanceof S.IdExpression);
assert(first("#pragma\nint x;").text === null);
sameText("#if defined(A) && B > 1\nint a;\n#elif !defined(C)\nint c;\n#else\nint d;\n#endif");
sameText("#ifdef A\n#elifdef B\nint b;\n#elifndef C\n#endif\n#ifndef D\n#endif", null, new CcppStandard(2023));
sameText("#if defined A\n#endif", "#if defined(A)\n#endif");
sameText("struct S {\n#ifdef A\n    int a;\n#endif\n};");
sameText("enum E {\n    A,\n#ifdef B\n    B,\n#endif\n    C\n};");
sameText("void f() {\n#if X\n    a();\n#endif\n}");
sameText("// line\n/* block */\nint x; // trailing\nvoid f() {\n    // inside\n}");
equal(parse("/* a */\n// b\r\n").items.map((c: any) => [c.block, c.text]), [[true, " a "], [false, " b"]]);
sameText("void f() {\n    switch (x) {\n        case 1: // one\n            a();\n    }\n}",
  "void f() {\n    switch (x) {\n        case 1:\n            a();\n            // one\n    }\n}");
console.log("ok");

## PRS-11 · Modules, `export` and `extern template`, which the pre-pass puts back

In [ ]:
unit = sameText("module;\n#include \"a.h\"\nexport module a.b:c;\nimport std;\nexport import :d;\nimport <vector>;\n"
  + "import \"e.h\";\nimport f.g:h;\nexport int x;\nexport {\n    int y;\n}\nexport namespace n {}\nmodule :private;");
equal(kinds(unit.items), ["GlobalModuleFragment", "IncludeDirective", "ModuleDeclaration", "ImportDeclaration",
  "ImportDeclaration", "ImportDeclaration", "ImportDeclaration", "ImportDeclaration", "ExportDeclaration",
  "ExportDeclaration", "ExportDeclaration", "PrivateModuleFragment"]);
const items = unit.items as any[];
equal([items[2].export, items[2].name, items[2].partition], [true, "a.b", "c"]);
equal([items[4].name, items[4].partition, items[5].system, items[6].system], [null, "d", true, false]);
sameText("export\nmodule\n    m\n;", "export module m;");
sameText('int x = module + import; // export module m;\nconst char *s = "import x;";\nint y = R"(module)";');
sameText("int f(int module);\nvoid g() {\n    import(1);\n    module.x = 2;\n    export_ = 3;\n}");
sameText("struct S {\n    template <class T>\n    void f();\n};\nextern template void S::f<int>();");
equal(kinds(parse("#define M a \\\n  b\nexport int x;").items), ["DefineDirective", "ExportDeclaration"]);
fails("int x; /* open", "syntax error", 1, 8);
fails("import <vector", "expected ';'", 1, 15);
fails('auto s = R"d(x', "syntax error", 1, 1);
fails('auto s = R"d', "syntax error", 1, 1);
fails("int x; export", "expected ;", 1, 14);
fails("export module m", "expected ';'", 1, 16);
fails("module :p;", "expected a module name after module", 1, 1);
fails("export module;", "expected a module name after module", 1, 8);
fails("module a.;", "expected a module name after module", 1, 1);
fails("module 3a;", "expected a module name after module", 1, 1);
fails("import;", "expected a module name after import", 1, 1);
fails('import "a.h" b;', "expected a module name after import", 1, 1);
fails("export module m [[a]];", "attributes of module declarations are not supported", 1, 8);
fails("import <a> [[x]];", "attributes of import declarations are not supported", 1, 1);
console.log("ok");

## PRS-12 · Errors name their line and column, counted in characters

In [ ]:
fails("int x = ;", "syntax error", 1, 7);
fails("int x", "expected ;", 1, 6);
fails("int é = 1;\nint ü", "expected ;", 2, 6);
fails("/* 🙂 */ int x", "expected ;", 1, 14); // an astral character is one character
fails("int x = static_cast<int>(1, 2);", "static_cast takes one operand", 1, 25);
fails("auto f = [](auto x) requires C<x> {};\nconcept C = true;",
  "LambdaDeclarator.requires needs C++20, but this is C++17", 1, 12, Ccpp17); // the first in source order
fails("void f() {\n    if consteval {}\n}", "syntax error", 2, 5);
fails("int x = a <=> b;", "BinaryExpression.operator '<=>' needs C++20, but this is C++17", 1, 9, Ccpp17);
fails("#warning w", "#warning needs C++23, but this is C++20", 1, 1, Ccpp20);
fails("int x = _Generic(1, int: 2);", "GenericSelection is not C++, but this is C++26", 1, 9);
fails("void f() throw(int);", "ThrowSpecifier with types was removed in C++17, but this is C++17", 1, 10, Ccpp17);
assert((parse("register int x;", C23).items[0] as any).specifiers[0].keyword === "register");
console.log("ok");

## PRS-13 · C: generic selections, compound literals and designators

In [ ]:
sameText("int x = _Generic(y, int: 1, char*: 2, default: 3);", null, C23);
const generic = initializer("int x = _Generic(y, int: 1, default: 3);", C23);
assert(generic.associations[0].type !== null && generic.associations[1].type === null);
sameText("struct P p = (struct P){1, 2};", null, C23);
assert(initializer("struct P p = (struct P){1, 2};", C23) instanceof S.CompoundLiteralExpression);
sameText("int a[] = {[1] = 2, [3 ... 4] = 5, .x = 6, .y.z = 7};", null, C23);
sameText("struct S s = {x: 1};", "struct S s = {.x = 1};", C23);
sameText("_Bool b;\nint *restrict p;\n_Noreturn void f(void);\n_Atomic int a;\n_Alignas(8) int c;", null, C23);
console.log("ok");